In [0]:
spark.sql("CREATE DATABASE IF NOT EXISTS ecommerce_bronze")
spark.sql("CREATE DATABASE IF NOT EXISTS ecommerce_silver")
spark.sql("CREATE DATABASE IF NOT EXISTS ecommerce_gold")
spark.sql("CREATE DATABASE IF NOT EXISTS ecommerce_audit")
spark.sql("CREATE DATABASE IF NOT EXISTS ecommerce_landing")

DataFrame[]

In [0]:
%sql
SHOW DATABASES LIKE 'ecommerce_*';

databaseName
ecommerce_audit
ecommerce_bronze
ecommerce_gold
ecommerce_landing
ecommerce_silver


In [0]:
print("Databases created successfully")
for db in ["ecommerce_landing","ecommerce_bronze","ecommerce_silver","ecommerce_gold","ecommerce_audit"]:
    print(f" - {db}")

Databases created successfully
 - ecommerce_landing
 - ecommerce_bronze
 - ecommerce_silver
 - ecommerce_gold
 - ecommerce_audit


##### 2. Configuration
 These constants control how much synthetic data is generated:

 | Variable | Value | Purpose |
 |----------|-------|---------|
 | `NUM_CUSTOMERS` | 10,000 | Fake customer profiles (names, emails, cities across 15 Indian cities) |
 | `NUM_PRODUCTS` | 200 | Product catalog across 8 categories (Electronics, Clothing, Books, etc.) |
 | `NUM_ORDERS` | 50,000 | Orders with ~0.5% intentional duplicates (to test dedup in notebook 02) |
 | `DATE_START / DATE_END` | Jan–Dec 2025 | All order dates are randomly spread across this full year |

 These values are small enough to run on a single-node Community Edition cluster in under 3 minutes.

In [0]:
NUM_CUSTOMERS = 10000
NUM_PRODUCTS = 200
NUM_ORDERS = 50000
DATE_START = "2025-01-01"
DATE_END = "2025-12-31"

print("Configuration set. Using Unity Catalog managed tables -- no file paths needed.")


Configuration set. Using Unity Catalog managed tables -- no file paths needed.


##### 3. Generate Customers Data


In [0]:
import random
import uuid
from datetime import datetime, timedelta
from pyspark.sql.types import (
    StructType, StructField, StringType, IntegerType,
    DoubleType, TimestampType, DateType
)
from pyspark.sql import functions as F

random.seed(42)

CITIES = [
    ("Mumbai", "Maharashtra"), ("Delhi", "Delhi"), ("Bangalore", "Karnataka"),
    ("Chennai", "Tamil Nadu"), ("Hyderabad", "Telangana"), ("Pune", "Maharashtra"),
    ("Kolkata", "West Bengal"), ("Ahmedabad", "Gujarat"), ("Jaipur", "Rajasthan"),
    ("Lucknow", "Uttar Pradesh"), ("Surat", "Gujarat"), ("Nagpur", "Maharashtra"),
    ("Indore", "Madhya Pradesh"), ("Bhopal", "Madhya Pradesh"), ("Kochi", "Kerala"),
]

FIRST_NAMES = [
    "Aarav", "Vivaan", "Aditya", "Vihaan", "Arjun", "Sai", "Reyansh", "Ayaan",
    "Krishna", "Ishaan", "Ananya", "Diya", "Aditi", "Myra", "Sara", "Aadhya",
    "Isha", "Riya", "Priya", "Neha", "Rohan", "Karan", "Amit", "Rahul", "Vikram"
]
LAST_NAMES = [
    "Sharma", "Verma", "Gupta", "Singh", "Kumar", "Patel", "Reddy", "Nair",
    "Iyer", "Joshi", "Mehta", "Shah", "Rao", "Das", "Mukherjee", "Kamat"
]
DOMAINS = ["gmail.com", "yahoo.com", "outlook.com", "hotmail.com"]

def generate_customers(n):
    customers = []
    for i in range(n):
        cid = f"CUST-{str(i+1).zfill(6)}"
        first = random.choice(FIRST_NAMES)
        last = random.choice(LAST_NAMES)
        city, state = random.choice(CITIES)
        email = f"{first.lower()}.{last.lower()}{random.randint(1,999)}@{random.choice(DOMAINS)}"
        phone = f"+91-{random.randint(70000,99999)}{random.randint(10000,99999)}"
        age = random.randint(18, 65)
        signup_date = datetime(2024, 1, 1) + timedelta(days=random.randint(0, 365))

        if random.random() < 0.03:
            email = None
        if random.random() < 0.02:
            phone = None

        customers.append((cid, first, last, email, phone, age, city, state,
                         signup_date.strftime("%Y-%m-%d"), "active" if random.random() > 0.1 else "inactive"))
    return customers

customer_data = generate_customers(NUM_CUSTOMERS)

customer_schema = StructType([
    StructField("customer_id", StringType(), False),
    StructField("first_name", StringType(), True),
    StructField("last_name", StringType(), True),
    StructField("email", StringType(), True),
    StructField("phone", StringType(), True),
    StructField("age", IntegerType(), True),
    StructField("city", StringType(), True),
    StructField("state", StringType(), True),
    StructField("signup_date", StringType(), True),
    StructField("status", StringType(), True),
])

df_customers = spark.createDataFrame(customer_data, schema=customer_schema)
df_customers.write.mode("overwrite").format("delta").saveAsTable("ecommerce_landing.customers")

print(f"Generated {df_customers.count()} customers")
df_customers.show(5, truncate=False)


Generated 10000 customers
+-----------+----------+---------+--------------------------+--------------+---+---------+-----------+-----------+--------+
|customer_id|first_name|last_name|email                     |phone         |age|city     |state      |signup_date|status  |
+-----------+----------+---------+--------------------------+--------------+---+---------+-----------+-----------+--------+
|CUST-000001|Rohan     |Singh    |rohan.singh760@outlook.com|+91-7802439256|26 |Mumbai   |Maharashtra|2024-02-22 |inactive|
|CUST-000002|Myra      |Verma    |myra.verma96@yahoo.com    |+91-7762376237|56 |Mumbai   |Maharashtra|2024-01-14 |active  |
|CUST-000003|Myra      |Nair     |myra.nair604@outlook.com  |+91-9652310851|28 |Ahmedabad|Gujarat    |2024-12-23 |active  |
|CUST-000004|Vikram    |Mehta    |vikram.mehta95@hotmail.com|+91-7316957052|40 |Delhi    |Delhi      |2024-11-05 |active  |
|CUST-000005|Vihaan    |Rao      |vihaan.rao566@outlook.com |+91-9717792397|57 |Delhi    |Delhi      |2024

##### 4. Generate Products Data

In [0]:
CATEGORIES = {
    "Electronics": [("Smartphone", 8999, 79999), ("Laptop", 25999, 149999), ("Headphones", 499, 14999),
                    ("Tablet", 9999, 59999), ("Smartwatch", 1999, 29999)],
    "Clothing": [("T-Shirt", 299, 2999), ("Jeans", 799, 4999), ("Jacket", 999, 7999),
                 ("Dress", 599, 5999), ("Sneakers", 999, 8999)],
    "Home & Kitchen": [("Mixer", 1499, 5999), ("Cookware Set", 999, 7999), ("Bedsheet", 399, 2999),
                       ("Lamp", 499, 3999), ("Water Purifier", 4999, 19999)],
    "Books": [("Fiction", 149, 699), ("Non-Fiction", 199, 999), ("Technical", 299, 1499),
              ("Comics", 99, 499), ("Self-Help", 149, 799)],
    "Sports": [("Cricket Bat", 999, 9999), ("Football", 399, 2999), ("Yoga Mat", 299, 1999),
               ("Running Shoes", 1999, 8999), ("Dumbbells", 499, 4999)],
    "Beauty": [("Sunscreen", 199, 999), ("Shampoo", 149, 799), ("Perfume", 499, 4999),
               ("Face Wash", 99, 599), ("Hair Oil", 79, 499)],
    "Grocery": [("Rice 5kg", 249, 599), ("Oil 1L", 99, 249), ("Tea 500g", 149, 499),
                ("Spices Pack", 99, 399), ("Dry Fruits 1kg", 499, 1999)],
    "Toys": [("Board Game", 299, 1999), ("Action Figure", 199, 1499), ("Puzzle", 99, 699),
             ("Building Blocks", 399, 2999), ("Remote Car", 499, 3999)],
}

def generate_products(n):
    products = []
    pid = 0
    while pid < n:
        for category, items in CATEGORIES.items():
            for name, min_price, max_price in items:
                if pid >= n:
                    break
                pid += 1
                product_id = f"PROD-{str(pid).zfill(4)}"
                price = round(random.uniform(min_price, max_price), 2)
                rating = round(random.uniform(2.5, 5.0), 1)
                stock = random.randint(0, 500)
                products.append((product_id, name, category, price, rating, stock,
                                random.choice(["active", "active", "active", "discontinued"])))
    return products[:n]

product_data = generate_products(NUM_PRODUCTS)

product_schema = StructType([
    StructField("product_id", StringType(), False),
    StructField("product_name", StringType(), True),
    StructField("category", StringType(), True),
    StructField("price", DoubleType(), True),
    StructField("rating", DoubleType(), True),
    StructField("stock_quantity", IntegerType(), True),
    StructField("status", StringType(), True),
])

df_products = spark.createDataFrame(product_data, schema=product_schema)
df_products.write.mode("overwrite").format("delta").saveAsTable("ecommerce_landing.products")

print(f"Generated {df_products.count()} products")
df_products.show(5, truncate=False)


Generated 200 products
+----------+------------+-----------+--------+------+--------------+------+
|product_id|product_name|category   |price   |rating|stock_quantity|status|
+----------+------------+-----------+--------+------+--------------+------+
|PROD-0001 |Smartphone  |Electronics|40729.29|4.7   |306           |active|
|PROD-0002 |Laptop      |Electronics|58038.4 |3.8   |326           |active|
|PROD-0003 |Headphones  |Electronics|3398.81 |3.0   |206           |active|
|PROD-0004 |Tablet      |Electronics|57171.71|4.8   |260           |active|
|PROD-0005 |Smartwatch  |Electronics|24243.77|2.5   |50            |active|
+----------+------------+-----------+--------+------+--------------+------+
only showing top 5 rows


##### 5. Generate Orders & Order Items Data

In [0]:
import json
from datetime import date

ORDER_STATUSES = ["completed", "completed", "completed", "completed",
                  "shipped", "shipped", "processing", "cancelled", "returned"]
PAYMENT_METHODS = ["UPI", "UPI", "Credit Card", "Debit Card", "Net Banking", "COD", "Wallet"]

start_date = date(2025, 1, 1)
end_date = date(2025, 12, 31)
date_range = (end_date - start_date).days

def generate_orders_and_items(num_orders, customer_ids, product_ids_prices):
    orders = []
    items = []
    item_id = 0

    for i in range(num_orders):
        oid = f"ORD-{str(i+1).zfill(7)}"
        cid = random.choice(customer_ids)
        order_date = start_date + timedelta(days=random.randint(0, date_range))
        status = random.choice(ORDER_STATUSES)
        payment = random.choice(PAYMENT_METHODS)

        num_items = random.choices([1, 2, 3, 4, 5], weights=[40, 30, 15, 10, 5])[0]
        order_total = 0.0
        selected_products = random.sample(product_ids_prices, min(num_items, len(product_ids_prices)))

        for prod_id, prod_price in selected_products:
            item_id += 1
            qty = random.choices([1, 2, 3], weights=[60, 30, 10])[0]
            discount = round(random.choice([0, 0, 0, 5, 10, 15, 20]) / 100.0, 2)
            line_total = round(prod_price * qty * (1 - discount), 2)
            order_total += line_total
            items.append((f"ITEM-{str(item_id).zfill(8)}", oid, prod_id,
                         qty, prod_price, discount, line_total))

        shipping = round(random.choice([0, 0, 49, 99, 149]), 2) if order_total < 499 else 0.0
        orders.append((oid, cid, str(order_date), status, payment,
                       round(order_total, 2), shipping, round(order_total + shipping, 2)))

        if random.random() < 0.005:
            orders.append((oid, cid, str(order_date), status, payment,
                           round(order_total, 2), shipping, round(order_total + shipping, 2)))

    return orders, items

customer_ids = [c[0] for c in customer_data]
product_ids_prices = [(p[0], p[3]) for p in product_data]

order_data, item_data = generate_orders_and_items(NUM_ORDERS, customer_ids, product_ids_prices)

order_schema = StructType([
    StructField("order_id", StringType(), False),
    StructField("customer_id", StringType(), True),
    StructField("order_date", StringType(), True),
    StructField("status", StringType(), True),
    StructField("payment_method", StringType(), True),
    StructField("subtotal", DoubleType(), True),
    StructField("shipping_fee", DoubleType(), True),
    StructField("total_amount", DoubleType(), True),
])

item_schema = StructType([
    StructField("item_id", StringType(), False),
    StructField("order_id", StringType(), True),
    StructField("product_id", StringType(), True),
    StructField("quantity", IntegerType(), True),
    StructField("unit_price", DoubleType(), True),
    StructField("discount_pct", DoubleType(), True),
    StructField("line_total", DoubleType(), True),
])

df_orders = spark.createDataFrame(order_data, schema=order_schema)
df_items = spark.createDataFrame(item_data, schema=item_schema)

df_orders.write.mode("overwrite").format("delta").saveAsTable("ecommerce_landing.orders")
df_items.write.mode("overwrite").format("delta").saveAsTable("ecommerce_landing.order_items")

print(f"Generated {df_orders.count()} orders (incl. ~0.5% intentional duplicates)")
print(f"Generated {df_items.count()} order line items")
df_orders.show(5, truncate=False)
df_items.show(5, truncate=False)

Generated 50255 orders (incl. ~0.5% intentional duplicates)
Generated 105115 order line items
+-----------+-----------+----------+----------+--------------+--------+------------+------------+
|order_id   |customer_id|order_date|status    |payment_method|subtotal|shipping_fee|total_amount|
+-----------+-----------+----------+----------+--------------+--------+------------+------------+
|ORD-0000001|CUST-005271|2025-05-30|completed |Debit Card    |1484.7  |0.0         |1484.7      |
|ORD-0000002|CUST-005603|2025-08-21|shipped   |COD           |2915.74 |0.0         |2915.74     |
|ORD-0000003|CUST-003709|2025-02-27|processing|Net Banking   |547.88  |0.0         |547.88      |
|ORD-0000004|CUST-007381|2025-06-20|completed |UPI           |22416.57|0.0         |22416.57    |
|ORD-0000005|CUST-003134|2025-04-23|cancelled |COD           |2361.37 |0.0         |2361.37     |
+-----------+-----------+----------+----------+--------------+--------+------------+------------+
only showing top 5 rows


##### 6.Verify Landing Zone Tables

In [0]:
print("Landing Zone Tables:")
for table in ["customers", "products", "orders", "order_items"]:
    count = spark.table(f"ecommerce_landing.{table}").count()
    print(f"  ecommerce_landing.{table} -> {count:,} rows")

print("\nSetup complete! Proceed to notebook 01_bronze_ingestion.")


Landing Zone Tables:
  ecommerce_landing.customers -> 10,000 rows
  ecommerce_landing.products -> 200 rows
  ecommerce_landing.orders -> 50,255 rows
  ecommerce_landing.order_items -> 105,115 rows

Setup complete! Proceed to notebook 01_bronze_ingestion.


##### 7. Row Count Verification & Explanation

 All four landing-zone counts are **correct and consistent** with the generation logic.

###### Expected vs. Actual

 | Table | Reported | Expected | Status |
 |-------|----------|----------|--------|
 | `customers` | 10,000 | `NUM_CUSTOMERS = 10000` | Exact match |
 | `products` | 200 | `NUM_PRODUCTS = 200` (sliced via `products[:n]`) | Exact match |
 | `orders` | 50,255 | `NUM_ORDERS = 50000` + ~0.5% duplicates &asymp; **50,250** | Matches (255 dups vs. ~250 expected) |
 | `order_items` | 105,115 | `NUM_ORDERS &times; avg items/order` &asymp; **105,000** | Matches |

###### Why `orders` > 50,000

 In Section 5, the generator intentionally re-appends ~0.5% of orders as duplicates:

 ```python
 if random.random() < 0.005:
     orders.append((oid, cid, str(order_date), status, payment,
                    round(order_total, 2), shipping, round(order_total + shipping, 2)))
 ```

 This produces **50,000 &times; 0.005 &asymp; 250** extra rows. The actual 255 is within statistical noise (and is deterministic given `random.seed(42)`). This is **by design** so notebook `02` can demonstrate dedup logic.

###### Why `order_items` &asymp; 105,000

 The number of items per order is sampled from `[1, 2, 3, 4, 5]` with weights `[40, 30, 15, 10, 5]`:

 $$
 \text{expected items per order} = \frac{1(40) + 2(30) + 3(15) + 4(10) + 5(5)}{100} = \frac{210}{100} = 2.10
 $$

 So expected items &asymp; `50,000 &times; 2.10 = 105,000`. Actual = 105,115 &mdash; matches.

 > **Note:** Order duplicates do **not** generate duplicate items &mdash; only the order row is re-appended, so `order_items` stays tied to the original 50,000 orders. That's why `order_items` count is based on `NUM_ORDERS`, not on the duplicated order count.
